# M10 — EDA: English popularity vs. Czech bestseller status

**Question (from supervisor):** Are bestsellers on the English market (books with many Goodreads ratings) automatically bestsellers in Czech, and vice versa — are there Czech bestsellers that were *not* popular in their original language?

**Why this matters for the thesis:** If foreign popularity strongly predicts Czech bestseller status, our model is redundant — a publisher could just sort foreign books by `gr_ratings_count` and pick the top. If the relationship is weak, it justifies the need for a multivariate model that combines popularity with genre, language, and timing signals.

**Data sources:**
- `data/interim/matched_dataset.csv` — all 225,483 NKC records with cascade match info
- `data/interim/training_dataset.csv` — 26,090 books after filters (≥ 2003, matched, ≥ 10 GR ratings, dedup)

Most analysis uses `training_dataset.csv` (clean reference cohort). We use `matched_dataset.csv` only for one cross-check.

Note: `gr_ratings_count` here is the *total* count from the 2017 Goodreads snapshot — same metric the supervisor is asking about. It's allowed as an EDA variable but NOT as a model feature (excluded from `X_train.csv` per project instruction about temporal leakage).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

REPO    = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = REPO / "data" / "interim"

train = pd.read_csv(INTERIM / "training_dataset.csv", dtype=str, keep_default_na=False)

# Numeric conversions
train["gr_ratings_count"]  = pd.to_numeric(train["gr_ratings_count"],  errors="coerce").fillna(0).astype(int)
train["sckn_appearances"]  = pd.to_numeric(train["sckn_appearances"],  errors="coerce").fillna(0).astype(int)
train["sckn_best_rank"]    = pd.to_numeric(train["sckn_best_rank"],    errors="coerce")
train["is_positive"]       = (train["sckn_appearances"] >= 1).astype(int)

print(f"Training cohort: {len(train):,} rows")
print(f"  SCKN positives: {train['is_positive'].sum():,} ({train['is_positive'].mean():.1%})")
print(f"  gr_ratings_count: min={train['gr_ratings_count'].min():,}  "
      f"median={int(train['gr_ratings_count'].median()):,}  "
      f"max={train['gr_ratings_count'].max():,}")

Training cohort: 26,090 rows
  SCKN positives: 1,405 (5.4%)
  gr_ratings_count: min=10  median=175  max=2,099,680


## 1 — Correlation: English popularity ↔ Czech bestseller status

Direct Pearson and Spearman correlations between Goodreads ratings count and SCKN appearances.

In [2]:
pearson  = train[["gr_ratings_count", "sckn_appearances", "is_positive"]].corr(method="pearson")
spearman = train[["gr_ratings_count", "sckn_appearances", "is_positive"]].corr(method="spearman")

print("Pearson correlations (linear):")
print(pearson.round(4).to_string())
print()
print("Spearman correlations (rank — more appropriate for heavy-tailed counts):")
print(spearman.round(4).to_string())

Pearson correlations (linear):
                  gr_ratings_count  sckn_appearances  is_positive
gr_ratings_count            1.0000            0.1496       0.0858
sckn_appearances            0.1496            1.0000       0.3723
is_positive                 0.0858            0.3723       1.0000

Spearman correlations (rank — more appropriate for heavy-tailed counts):
                  gr_ratings_count  sckn_appearances  is_positive
gr_ratings_count            1.0000            0.0625       0.0615
sckn_appearances            0.0625            1.0000       0.9995
is_positive                 0.0615            0.9995       1.0000


**Interpretation:** Pearson on heavy-tailed counts is misleading (dominated by outliers). Spearman is the honest number — it asks "do higher Goodreads ranks correspond to higher SCKN ranks?" without assuming linearity.

If `gr_ratings_count` × `is_positive` Spearman is **above 0.4** → strong relationship (foreign popularity → Czech popularity). If **below 0.2** → weak relationship, justifies our multivariate approach.

## 2 — Top-N analysis: does English popularity predict Czech success?

Sort training set by `gr_ratings_count` descending. For each top-N bucket, compute the fraction that are SCKN positive. Baseline = 5.4% (positive rate in training).

In [3]:
sorted_by_gr = train.sort_values("gr_ratings_count", ascending=False).reset_index(drop=True)
baseline = train["is_positive"].mean()

results = []
for n in [50, 100, 250, 500, 1000, 2500, 5000, 10000, len(train)]:
    top_n = sorted_by_gr.head(n)
    pos_count = top_n["is_positive"].sum()
    pos_rate = top_n["is_positive"].mean()
    lift = pos_rate / baseline
    results.append({
        "top_N":           n,
        "min_gr_ratings":  int(top_n["gr_ratings_count"].min()),
        "n_SCKN_positive": int(pos_count),
        "pct_positive":    f"{pos_rate:.1%}",
        "lift_vs_baseline": f"{lift:.2f}x",
    })

topN_table = pd.DataFrame(results)
print(f"Baseline positive rate (all training): {baseline:.1%}")
print()
print(topN_table.to_string(index=False))

Baseline positive rate (all training): 5.4%

 top_N  min_gr_ratings  n_SCKN_positive pct_positive lift_vs_baseline
    50          230866               18        36.0%            6.68x
   100          116481               29        29.0%            5.39x
   250           53467               52        20.8%            3.86x
   500           29333               80        16.0%            2.97x
  1000           14592              120        12.0%            2.23x
  2500            4761              249        10.0%            1.85x
  5000            1459              401         8.0%            1.49x
 10000             330              678         6.8%            1.26x
 26090              10             1405         5.4%            1.00x


**Reading this table:**
- If top-100 books by English popularity have e.g. 40% SCKN positive (7-8× baseline) → English popularity is a strong predictor.
- If top-100 have ~10% (2× baseline) → modest predictor, lots of false positives.
- The `lift_vs_baseline` column is the practical metric: "how many times better than random is this filter?"

## 3 — Reverse view: where do SCKN positives sit on the Goodreads popularity scale?

Among the 1,405 SCKN positives, what's the distribution of `gr_ratings_count`? If many positives have low Goodreads counts → there exist "Czech bestsellers that flew under the English radar".

In [4]:
positives = train[train["is_positive"] == 1].copy()
print(f"SCKN positives: {len(positives):,}")
print()
print("gr_ratings_count distribution among SCKN positives:")
print(positives["gr_ratings_count"].describe(percentiles=[.1, .25, .5, .75, .9, .95]).round(0).to_string())

SCKN positives: 1,405

gr_ratings_count distribution among SCKN positives:
count       1405.0
mean       16725.0
std       130005.0
min           10.0
10%           28.0
25%           71.0
50%          309.0
75%         2086.0
90%        11597.0
95%        34108.0
max      2099680.0


In [5]:
# How many positives are in different popularity buckets?
buckets = [
    (10, 100,       "obscure (10–99 ratings)"),
    (100, 1000,     "niche (100–999)"),
    (1000, 10000,   "recognized (1k–10k)"),
    (10000, 100000, "popular (10k–100k)"),
    (100000, 10**8, "viral (100k+)"),
]

print("SCKN positives by Goodreads popularity bucket:")
for lo, hi, label in buckets:
    mask = (positives["gr_ratings_count"] >= lo) & (positives["gr_ratings_count"] < hi)
    n = mask.sum()
    pct = n / len(positives) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label:<28} : {n:>4} ({pct:>5.1f}%)  {bar}")

SCKN positives by Goodreads popularity bucket:
  obscure (10–99 ratings)      :  411 ( 29.3%)  ██████████████
  niche (100–999)              :  537 ( 38.2%)  ███████████████████
  recognized (1k–10k)          :  296 ( 21.1%)  ██████████
  popular (10k–100k)           :  128 (  9.1%)  ████
  viral (100k+)                :   33 (  2.3%)  █


## 4 — Concrete examples

### 4a — English bestsellers that flopped in Czech
Books with very high `gr_ratings_count` but `sckn_appearances = 0` — "English mass-market hits that didn't translate to Czech bestseller status."

In [6]:
english_hits_czech_flops = (
    train[train["is_positive"] == 0]
    .sort_values("gr_ratings_count", ascending=False)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "gr_ratings_count", "sckn_appearances"]]
)
print("TOP 20 high-Goodreads-popularity books that were NEVER in SCKN top 10:")
english_hits_czech_flops

TOP 20 high-Goodreads-popularity books that were NEVER in SCKN top 10:


,author,original_title,czech_title,czech_pub_year,source_lang,gr_ratings_count,sckn_appearances
1652,"Shakespeare, William",Romeo and Juliet,Romeo a Julie,2004,eng,1656919,0
1415,"Sparks, Nicholas",Notebook,Zápisník jedné lásky,2004,eng,1064723,0
3315,"Picoult, Jodi",My sister's keeper,Je to i můj život,2005,eng,876319,0
4386,"Weisberger, Lauren",Devil wears Prada,Ďábel nosí Pradu,2006,eng,675927,0
2678,"Grisham, John",Time to kill,--a je čas zabíjet,2005,eng,604739,0
2476,"Shakespeare, William",Hamlet,Hamlet,2005,eng,526122,0
1846,"Chevalier, Tracy",Girl with a pearl earring,Dívka s perlou,2004,eng,481621,0
1925,"Moore, Alan",Watchmen,Strážci,2004,eng,406669,0
276,"McCourt, Frank",Angela's ashes,Andělin popel,2003,eng,401029,0
1006,"Colfer, Eoin",Artemis Fowl. The eternity code,Artemis Fowl,2003,eng,393254,0


### 4b — Czech bestsellers with low Goodreads popularity
SCKN positives with the *lowest* `gr_ratings_count` — books that were Czech hits but barely registered on the English-language Goodreads.

In [7]:
czech_hits_low_english = (
    positives
    .sort_values("gr_ratings_count", ascending=True)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "gr_ratings_count", "sckn_appearances", "sckn_best_rank"]]
)
print("TOP 20 SCKN-positive books with the LOWEST Goodreads ratings count:")
czech_hits_low_english

TOP 20 SCKN-positive books with the LOWEST Goodreads ratings count:


,author,original_title,czech_title,czech_pub_year,source_lang,gr_ratings_count,sckn_appearances,sckn_best_rank
15599,"Smith, Wilbur A",Desert God,Řeka bohů,2014,eng,10,6,4.0
13137,"Owen, Mark",No easy day,Nelehký den,2013,eng,10,2,5.0
23286,"Garnier, Stéphane",Agir et penser comme un chat,Chovejte se jako kočka,2018,fre,10,16,2.0
5333,"Sapkowski, Andrzej",Czas pogardy,Zaklínač,2011,pol,10,1,10.0
18227,"Bass, Guy",Beast of Grubbers Nubbin,Záplaťák,2016,eng,10,1,2.0
18060,Áslaug Jónsdóttir,Nei! sagði litla skrímslið,Ne! řeklo strašidýlko,2016,ice,10,1,7.0
240,"Klemperer, Victor","LTI, Notizbuch eines Philologen",Jazyk Třetí říše - LTI,2003,ger,10,1,5.0
15217,"Gaarder, Jostein",Julemysteriet,Kouzelný kalendář,2014,nor,11,1,6.0
6973,"Barbery, Muriel",Élégance du hérisson,S elegancí ježka,2008,fre,11,16,1.0
10724,"Sem-Sandberg, Steve",Fattiga i Łodź,Chudí v Lodži,2011,swe,11,4,3.0


### 4c — Both ends together (the "obvious matches")
Books in the top quartile of `gr_ratings_count` that ALSO are SCKN positive — the easy predictions where English popularity does carry over.

In [8]:
obvious = (
    positives
    .sort_values("gr_ratings_count", ascending=False)
    .head(20)
    [["author", "original_title", "czech_title", "czech_pub_year",
      "source_lang", "gr_ratings_count", "sckn_appearances", "sckn_best_rank"]]
)
print("TOP 20 SCKN-positive books with the HIGHEST Goodreads ratings count:")
obvious

TOP 20 SCKN-positive books with the HIGHEST Goodreads ratings count:


,author,original_title,czech_title,czech_pub_year,source_lang,gr_ratings_count,sckn_appearances,sckn_best_rank
530,"Tolkien, J. R. R",Hobbit,"Hobit, aneb, Cesta tam a zase zpátky",2003,eng,2099680,1,10.0
12190,"Roth, Veronica",Divergent,Divergence,2012,eng,1962813,14,2.0
891,"Rowling, J. K",Harry Potter and the prisoner of Azkaban,Harry Potter a vězeň z Azkabanu,2003,eng,1876252,30,2.0
1004,"Rowling, J. K",Harry Potter and the goblet of fire,Harry Potter a ohnivý pohár,2003,eng,1792561,32,2.0
1331,"Rowling, J. K",Harry Potter and the Order of the Phoenix,Harry Potter a Fénixův řád,2004,eng,1766895,92,1.0
2952,"Lewis, C. S","Lion, the witch and the wardrobe",[Letopisy Narnie],2005,eng,1575387,31,1.0
16685,"Hawkins, Paula",Girl on the train,Dívka ve vlaku,2015,eng,1076144,31,1.0
12688,"Roth, Veronica",Insurgent,Rezistence,2012,eng,849014,10,2.0
4311,"Dahl, Roald",Charlie and the chocolate factory,Karlík a továrna na čokoládu,2006,eng,462516,20,1.0
4597,"Martin, George R. R",Feast for crows,Píseň ledu a ohně,2012,eng,437398,2,3.0


## 5 — Conditional probability tables

For practical decision-making: given a popularity threshold, what's the conditional probability of being a SCKN positive?

    P(SCKN+ | gr_ratings ≥ T) = (positives with gr_ratings ≥ T) / (all books with gr_ratings ≥ T)

In [9]:
thresholds = [100, 500, 1000, 5000, 10000, 50000, 100000, 500000, 1_000_000]

rows = []
for T in thresholds:
    mask = train["gr_ratings_count"] >= T
    n_total = mask.sum()
    n_pos   = train.loc[mask, "is_positive"].sum()
    p_cond  = n_pos / n_total if n_total > 0 else 0
    lift    = p_cond / baseline if baseline > 0 else 0
    rows.append({
        "gr_ratings ≥": f"{T:>9,}",
        "n_books_in_pool": n_total,
        "n_SCKN_positive": int(n_pos),
        "P(SCKN+ | pool)": f"{p_cond:.1%}",
        "lift_vs_baseline": f"{lift:.2f}x",
    })

print(f"Baseline P(SCKN+) on whole training: {baseline:.1%}")
print()
print(pd.DataFrame(rows).to_string(index=False))

Baseline P(SCKN+) on whole training: 5.4%

gr_ratings ≥  n_books_in_pool  n_SCKN_positive P(SCKN+ | pool) lift_vs_baseline
         100            15975              994            6.2%            1.16x
         500             8402              583            6.9%            1.29x
       1,000             6052              457            7.6%            1.40x
       5,000             2396              241           10.1%            1.87x
      10,000             1380              161           11.7%            2.17x
      50,000              269               52           19.3%            3.59x
     100,000              130               33           25.4%            4.71x
     500,000               14                8           57.1%           10.61x
   1,000,000                9                7           77.8%           14.44x


## 6 — Summary for thesis discussion

Fill in the numbers from above:

1. **Correlation gr_ratings_count ↔ sckn_appearances (Spearman):** ___
2. **Top-100 by gr_ratings_count → % SCKN positive:** ___%  (vs. 5.4% baseline → lift ___x)
3. **% of SCKN positives that are "obscure" (gr_ratings < 1000):** ___%
4. **Concrete example, English bestseller that flopped in Czech:** ___
5. **Concrete example, Czech bestseller with low Goodreads count:** ___

**Pattern check:** if (2) is < 50% and (3) is > 10%, the data strongly supports the thesis framing: English popularity is a weak filter alone, and there are real "Czech-specific" success patterns the model can pick up that aren't captured by raw popularity.

These numbers go into:
- Reply to supervisor (Section: EDA support for the framing).
- Thesis Chapter on data analysis.
- Thesis Discussion (justifying why we need a multivariate model).